# 05d — No respuesta e imputación de ingresos en la era histórica

Diagnóstico, **no escribe capas**: solo `data/meta/imputacion_*.csv`. Decide (vos, no este
notebook) cómo tratar en 06c los hogares con ingresos imputados de 2003T3-2015T2.

**Contexto.** Desde 2009T3 INDEC imputa los ingresos no declarados por hot-deck aleatorio,
con reprocesamiento hacia atrás hasta 2003T3, y marca los registros con algún componente
imputado (`IDIMPH` hogar, `IDIMPP` persona). Desde 2016T2 vuelve a reponderar
(`PONDIH`/`PONDII`/`PONDIIO`) y deja el ingreso no declarado en 0 (`DECCFR==12`).

**Fuente de las definiciones**: diseño de registro INDEC 2009
(`data/EPH_diseno_registro_bases_2009.pdf`) — `IDIMPH` p. 12, `IDIMPP` pp. 28-29,
montos pp. 20-27, códigos especiales p. 30. Mapeo posición/dígito → montos en
`src/imputacion_ingresos.py` (`MAPEO_IDIMPP`):

| Pos. | Dígito | Variables | Tipo | Entra en |
|---|---|---|---|---|
| 1 | 1 | `PP08D1` | exacto si cumple I1 | ambas |
| 1 | 2 | `PP08D4+PP08F1+PP08F2` | exacto si cumple I1 | ambas |
| 1 | 3 | unión 1+2 (con I1a, `PP08D4 ⊂ PP08D1`, no se cuenta dos veces) | exacto si cumple I1 | ambas |
| 1 | 1/2/3 | respaldo `P21` si no cumple I1 | cota superior | max |
| 1 | 4 / 5 | `PP06C` / `PP06D` | exacto | ambas |
| 2 | 1 / 2 / 3 | `PP08J1` / `PP08J2+PP08J3` / unión (por fuera de `P21`, entran a `P47T`) | exacto | ambas |
| 3 | 1 | `TOT_P12` | exacto | ambas |
| 4 | 1 | `V2_M` | exacto | ambas |
| 4 | 2 | grupo amplio (`V21_M, V3_M, V4_M, V5_M, V8_M, V9_M, V10_M, V11_M, V12_M, V18_M, V19_AM`) | cota superior | max |
| 4 | 4 | residuo `T_VI − Σ V*_M` (≤0 → 0, "marcado sin monto") | [inferido] | max |
| 4 | 3/5/6/7 | unión de sus partes | — | por parte |
| 5 | 1 | `PJ1_1→P21`, `PJ2_1→TOT_P12`, `PJ3_1→V5_M`; sin `PJ*` → NA | [inferido] | max |

- **I1a**: `P21 == PP08D1+PP08F1+PP08F2` (±$1); **I1b**: I1a + `PP08D4`; `PP08J*` van por fuera de `P21`
  (verificado en la primera corrida de este notebook, ver "Anomalías");
  independiente: `P21 == PP06C` o `PP06D`.
- Códigos especiales: `-7`/`-8` → 0, `-9` → NA (p. 30).
- `fraccion_min` = solo componentes exactos / `ITF`; `fraccion_max` = exactos + cotas + [inferido]
  / `ITF`. `NA` si `ITF==0` o si un monto de esa fracción es NA.

In [ ]:
import sys
import time
from pathlib import Path


def _raiz_repo(desde: Path | None = None) -> Path:
    actual = (desde or Path.cwd()).resolve()
    for candidato in (actual, *actual.parents):
        if (candidato / "pytest.ini").exists():
            return candidato
    raise FileNotFoundError(f"no se encontró la raíz del repo subiendo desde {actual}")


REPO = _raiz_repo()
sys.path.insert(0, str(REPO / "src"))

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import canasta_retropolada as cr
import distancia_linea_pobreza
import imputacion_ingresos as ii
import panel
import perfil_hogar
import trayectoria_distancia
from armonizacion import trimestres_publicados

ARMONIZADO = REPO / "data" / "01_armonizado"
NUCLEO = REPO / "data" / "02_nucleo"
DISTANCIA = REPO / "data" / "02b_distancia_linea_pobreza" / "hogar"
PANEL_DIR = REPO / "data" / "03_panel"
TRAYECTORIA = REPO / "data" / "05_trayectoria_distancia"
META = REPO / "data" / "meta"
CEDLAS = REPO / "data" / "canastas_indec" / "retropolacion" / "cedlas" / "series_comparables_anexo_b.csv"
CODEBOOK_2009 = REPO / "data" / "EPH_diseno_registro_bases_2009.pdf"
assert CODEBOOK_2009.exists(), "falta el diseño de registro 2009 -- PARAR"

TRIMESTRES_HIST = [t for t in trimestres_publicados() if t <= (2015, 2)]
TRIMESTRES_REG = [t for t in trimestres_publicados() if t >= (2016, 2)]
AGLOMERADOS_EXCLUIDOS_COMPARABLE = {38, 91, 93}
UMBRALES = (0.1, 0.25, 0.5)
UMBRAL_POS5_MARGINAL = 0.5
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 220)
print(f"{len(TRIMESTRES_HIST)} trimestres históricos, {len(TRIMESTRES_REG)} regulares")

## 0. Procedencia de las bases históricas

Si los archivos del caché fueran versiones previas al reprocesamiento de 2009, `IDIMPH`
no existiría en 2003-2009 o la frecuencia de imputación saltaría en 2009T3 (se mira en el
paso 3). Fecha de la captura de Wayback de cada archivo, desde `manifiesto_insumos.csv`:

In [ ]:
manifiesto = pd.read_csv(META / "manifiesto_insumos.csv")
manifiesto = manifiesto[manifiesto["anio"] <= 2015].copy()
manifiesto["captura_wayback"] = manifiesto["url"].str.extract(r"/web/(\d{8})")[0]
manifiesto.groupby("captura_wayback").agg(n_archivos=("archivo", "size"), primero=("archivo", "first"),
                                           ultimo=("archivo", "last"))

## 1-2. Carga, verificación contra el codebook y fracción imputada

Un recorrido por trimestre histórico: lee `01_armonizado` (`IDIMPH`, `IDIMPP`, `PJ*` y las
variables de monto no están en `02_nucleo`), verifica códigos, consistencia
`IDIMPH`/`IDIMPP`, montos marcados, identidades I1-I3 y códigos especiales, y calcula
`fraccion_min`/`fraccion_max` por hogar. Las discrepancias se cuentan, no se corrigen.

In [ ]:
COLUMNAS_02B_HIST = ["CODUSU", "NRO_HOGAR", "ANO4", "TRIMESTRE", "peso_hogar", "n_miembros", "pobre", "indigente",
                     "ratio_linea_pobreza"]

filas_verif, filas_especiales, filas_disponibilidad, hogares_hist = [], [], [], []
t0 = time.time()
for anio, trimestre in TRIMESTRES_HIST:
    hogar = ii.leer_marcas_imputacion(anio, trimestre, ARMONIZADO, "hogar")
    individual = ii.leer_marcas_imputacion(anio, trimestre, ARMONIZADO, "individual",
                                           columnas_extra=["ESTADO", "CAT_OCUP", "PP07H"])
    columnas_nucleo_h = pq.read_schema(NUCLEO / "hogar" / f"ANO4={anio}" / f"TRIMESTRE={trimestre}" / "data.parquet").names
    columnas_nucleo_i = pq.read_schema(NUCLEO / "individual" / f"ANO4={anio}" / f"TRIMESTRE={trimestre}" / "data.parquet").names
    filas_disponibilidad.append({
        "ANO4": anio, "TRIMESTRE": trimestre,
        **{f"{v}_armonizado": v in (hogar.columns if v == "IDIMPH" else individual.columns)
           for v in ("IDIMPH", "IDIMPP", "PJ1_1", "PJ2_1", "PJ3_1")},
        "IDIMPH_nucleo": "IDIMPH" in columnas_nucleo_h, "IDIMPP_nucleo": "IDIMPP" in columnas_nucleo_i})

    pos_h = ii.descomponer_codigo_imputacion(hogar["IDIMPH"], ii.DIGITOS_VALIDOS_IDIMPH)
    pos_p = ii.descomponer_codigo_imputacion(individual["IDIMPP"])
    consistencia = ii.verificar_idimph_contra_idimpp(hogar, individual)
    marcado = ii.verificar_monto_marcado(individual)
    identidad = ii.verificar_identidad_ocupacion_principal(individual)
    i2, i3 = ii.verificar_identidades_ingreso(hogar, individual)
    fraccion = ii.fraccion_itf_imputada(hogar, individual)

    pos1_asalariado = pos_p["pos_1"].isin([1, 2, 3]).fillna(False)
    fila = {"ANO4": anio, "TRIMESTRE": trimestre, "n_hogares": len(hogar), "n_personas": len(individual),
            "n_idimph_invalido": int(pos_h["codigo_invalido"].sum()),
            "n_idimpp_invalido": int(pos_p["codigo_invalido"].sum()),
            "n_pos1_123_personas": int(pos1_asalariado.sum()),
            "pct_pos1_123_cumple_i1a": 100 * (identidad.loc[pos1_asalariado, "variante_i1"] == "a").mean(),
            "pct_pos1_123_cumple_i1b": 100 * (identidad.loc[pos1_asalariado, "variante_i1"] == "b").mean(),
            "pct_pos1_123_respaldo_p21": 100 * (~identidad.loc[pos1_asalariado, "cumple_i1"]).mean(),
            "pct_personas_i2": 100 * i2.mean(), "pct_hogares_i3": 100 * i3.mean()}
    for pos in ii.POSICIONES:
        fila[f"n_discrepa_idimph_pos{pos}"] = int(consistencia[f"discrepa_pos_{pos}"].fillna(False).sum())
        fila[f"n_pos{pos}_monto_cero"] = int(marcado[f"pos{pos}_marcado_monto_cero"].sum())
        fila[f"n_pos{pos}_monto_menos9"] = int(marcado[f"pos{pos}_marcado_monto_menos9"].sum())
        fila[f"n_pos{pos}_monto_na"] = int(marcado[f"pos{pos}_marcado_monto_na"].sum())
    fila["n_pos4_marcado_sin_monto"] = int(marcado["pos4_marcado_sin_monto"].sum())
    fila["n_pos5_sin_pj"] = int(marcado["pos5_sin_pj"].sum())
    filas_verif.append(fila)

    especiales = ii.contar_codigos_especiales(individual, ii.VARIABLES_MONTO)
    especiales["ANO4"], especiales["TRIMESTRE"] = anio, trimestre
    filas_especiales.append(especiales)

    situacion = perfil_hogar.calcular_situacion_laboral(individual)
    cat_ocup = pd.to_numeric(individual["CAT_OCUP"], errors="coerce")
    independiente = (pd.to_numeric(individual["ESTADO"], errors="coerce") == 1) & cat_ocup.isin([1, 2])
    hay_independiente = independiente.groupby([individual["CODUSU"], individual["NRO_HOGAR"]]).any().rename("hay_independiente")

    tabla = fraccion.copy()
    for pos in ii.POSICIONES:
        tabla[f"idimph_pos{pos}"] = pos_h[f"pos_{pos}"].to_numpy()
    tabla["ANO4"], tabla["TRIMESTRE"] = anio, trimestre
    tabla["AGLOMERADO"] = pd.to_numeric(hogar["AGLOMERADO"], errors="coerce").to_numpy()
    tabla["ITF"] = ii.limpiar_monto(hogar["ITF"]).to_numpy()
    tabla = tabla.merge(situacion, on=["CODUSU", "NRO_HOGAR"], how="left")
    tabla = tabla.merge(hay_independiente.reset_index(), on=["CODUSU", "NRO_HOGAR"], how="left")
    distancia = pd.read_parquet(DISTANCIA / f"ANO4={anio}" / f"TRIMESTRE={trimestre}" / "data.parquet",
                                columns=COLUMNAS_02B_HIST)
    tabla = tabla.merge(distancia, on=["CODUSU", "NRO_HOGAR", "ANO4", "TRIMESTRE"], how="left")
    hogares_hist.append(tabla)

hogares_hist = pd.concat(hogares_hist, ignore_index=True)
verificacion = pd.DataFrame(filas_verif)
codigos_especiales = pd.concat(filas_especiales, ignore_index=True)
disponibilidad = pd.DataFrame(filas_disponibilidad)
print(f"{len(hogares_hist):,} hogares históricos, {time.time() - t0:.0f}s")

### 1. Disponibilidad por capa

`IDIMPH`/`IDIMPP` existen solo en `01_armonizado` (no entran a `ESQUEMA_NUCLEO`); `PJ*`
solo en parte de los trimestres.

In [ ]:
disponibilidad.to_csv(META / "imputacion_disponibilidad.csv", index=False)
resumen_disp = disponibilidad.drop(columns=["ANO4", "TRIMESTRE"]).sum().rename("n_trimestres_con_variable")
display(resumen_disp.to_frame())
con_pj = disponibilidad[disponibilidad["PJ1_1_armonizado"]]
print(f"PJ* presentes: {len(con_pj)} trimestres, {con_pj['ANO4'].min()}T{con_pj.iloc[0]['TRIMESTRE']} -- "
      f"{con_pj['ANO4'].max()}T{con_pj.iloc[-1]['TRIMESTRE']}")

### 2. Verificación contra el codebook, por trimestre

Columnas: códigos fuera del rango definido, discrepancias `IDIMPH` contra el agregado de
`IDIMPP` por posición, personas con posición marcada y monto mapeado 0/-9/NA, "marcado sin
monto" de la pos. 4 dígito 4, pos. 5 sin `PJ*`, y cumplimiento de I1 (entre personas con
pos. 1 dígitos 1-3), I2 (`P47T`) e I3 (`ITF`).

In [ ]:
verificacion.to_csv(META / "imputacion_verificacion_trimestre.csv", index=False)
columnas_conteo = [c for c in verificacion.columns if c.startswith("n_") and c not in ("n_hogares", "n_personas")]
print("totales sobre los 47 trimestres:")
display(verificacion[columnas_conteo].sum().to_frame("total"))
display(verificacion[["ANO4", "TRIMESTRE", "n_pos1_123_personas", "pct_pos1_123_cumple_i1a",
                      "pct_pos1_123_cumple_i1b", "pct_pos1_123_respaldo_p21", "pct_personas_i2",
                      "pct_hogares_i3"]].round(2))

In [ ]:
codigos_especiales.to_csv(META / "imputacion_codigos_especiales.csv", index=False)
especiales_totales = codigos_especiales.groupby("variable")[["n_menos7", "n_menos8", "n_menos9", "n_na"]].sum()
display(especiales_totales[especiales_totales.sum(axis=1) > 0])
print("por trimestre (solo variables con algún código especial):")
display(codigos_especiales[codigos_especiales[["n_menos7", "n_menos8", "n_menos9", "n_na"]].sum(axis=1) > 0]
        .pivot_table(index=["ANO4", "TRIMESTRE"], columns="variable", values=["n_menos7", "n_menos8"]).head(10))

#### Pos. 5 (plan de empleo): frecuencia antes de seguir

Si la pos. 5 marcada supera el 0,5% de los hogares en algún trimestre, el notebook frena
acá (la pos. 5 sin `PJ*` deja `fraccion_max` en NA y contamina la comparación).

In [ ]:
pos5 = (hogares_hist.assign(pos5=hogares_hist["idimph_pos5"] == 1)
        .groupby(["ANO4", "TRIMESTRE"]).agg(pct_hogares_pos5=("pos5", "mean"),
                                             n_hogares_pos5=("pos5", "sum")))
pos5["pct_hogares_pos5"] *= 100
pos5 = pos5.merge(verificacion.set_index(["ANO4", "TRIMESTRE"])[["n_pos5_sin_pj"]], left_index=True, right_index=True)
pos5.to_csv(META / "imputacion_pos5_trimestre.csv")
maximo_pos5 = pos5["pct_hogares_pos5"].max()
print(f"máximo % de hogares con pos. 5 marcada: {maximo_pos5:.3f}% "
      f"({pos5['pct_hogares_pos5'].idxmax()})")
assert maximo_pos5 < UMBRAL_POS5_MARGINAL, "pos. 5 no marginal -- PARAR y avisar antes del paso 5"
display(pos5[pos5["n_hogares_pos5"] > 0].round(3))

#### `IMPUTA` (era regular)

Valores por trimestre (incluido si `NaN` reemplaza a `0`) y cruce, a nivel hogar, con
`DECCFR==12`/`PONDIH==0`, y a nivel persona con `P21==-9` y `PONDII==0`.

In [ ]:
filas_imputa, hogares_reg = [], []
for anio, trimestre in TRIMESTRES_REG:
    ind = ii.leer_marcas_imputacion(anio, trimestre, ARMONIZADO, "individual", columnas_extra=["PONDII", "PONDIIO"])
    nuc_h = pd.read_parquet(NUCLEO / "hogar" / f"ANO4={anio}" / f"TRIMESTRE={trimestre}" / "data.parquet",
                            columns=["CODUSU", "NRO_HOGAR", "AGLOMERADO", "DECCFR", "PONDIH", "ITF"])
    nuc_i = pd.read_parquet(NUCLEO / "individual" / f"ANO4={anio}" / f"TRIMESTRE={trimestre}" / "data.parquet",
                            columns=["CODUSU", "NRO_HOGAR", "ESTADO", "CAT_OCUP", "PP07H"])
    imputa = pd.to_numeric(ind["IMPUTA"], errors="coerce")
    p21_crudo = pd.to_numeric(ind["P21"], errors="coerce")
    pondii = pd.to_numeric(ind.get("PONDII"), errors="coerce") if "PONDII" in ind else pd.Series(np.nan, index=ind.index)
    imputa_hogar = (imputa == 1).groupby([ind["CODUSU"], ind["NRO_HOGAR"]]).any().rename("imputa_hogar").reset_index()
    nuc_h = nuc_h.merge(imputa_hogar, on=["CODUSU", "NRO_HOGAR"], how="left")
    dec12 = nuc_h["DECCFR"] == 12
    filas_imputa.append({
        "ANO4": anio, "TRIMESTRE": trimestre, "dtype_imputa": str(ind["IMPUTA"].dtype),
        "n_imputa_0": int((imputa == 0).sum()), "n_imputa_1": int((imputa == 1).sum()),
        "n_imputa_na": int(imputa.isna().sum()), "n_otros_valores": int((~imputa.isin([0, 1]) & imputa.notna()).sum()),
        "n_personas_imputa1_p21_menos9": int(((imputa == 1) & (p21_crudo == -9)).sum()),
        "n_personas_imputa1_pondii0": int(((imputa == 1) & (pondii == 0)).sum()),
        "n_hogares": len(nuc_h), "pct_hogares_deccfr12": 100 * dec12.mean(),
        "pct_hogares_pondih0": 100 * (nuc_h["PONDIH"] == 0).mean(),
        "n_hogares_imputa": int(nuc_h["imputa_hogar"].fillna(False).sum()),
        "n_hogares_imputa_y_deccfr12": int((nuc_h["imputa_hogar"].fillna(False) & dec12.fillna(False)).sum()),
    })
    situacion = perfil_hogar.calcular_situacion_laboral(nuc_i)
    independiente = (pd.to_numeric(nuc_i["ESTADO"], errors="coerce") == 1) & pd.to_numeric(nuc_i["CAT_OCUP"], errors="coerce").isin([1, 2])
    hay_ind = independiente.groupby([nuc_i["CODUSU"], nuc_i["NRO_HOGAR"]]).any().rename("hay_independiente").reset_index()
    nuc_h = nuc_h.merge(situacion, on=["CODUSU", "NRO_HOGAR"], how="left").merge(hay_ind, on=["CODUSU", "NRO_HOGAR"], how="left")
    nuc_h["ANO4"], nuc_h["TRIMESTRE"] = anio, trimestre
    hogares_reg.append(nuc_h)

imputa_regular = pd.DataFrame(filas_imputa)
hogares_reg = pd.concat(hogares_reg, ignore_index=True)
imputa_regular.to_csv(META / "imputacion_imputa_regular.csv", index=False)
display(imputa_regular.round(2))

## 3. Magnitud

% de hogares con alguna imputación (`IDIMPH != 00000`) y por posición, por trimestre
histórico, contra `DECCFR==12`/`PONDIH==0` en la era regular. % del `ITF` expandido
(`peso_hogar`) que es imputado, como intervalo.

In [ ]:
def _magnitud_trimestre(g: pd.DataFrame) -> pd.Series:
    w = g["peso_hogar"].astype(float)
    ambas_validas = g["fraccion_min"].notna() & g["fraccion_max"].notna()
    itf_validas = (w * g["ITF"].astype(float))[ambas_validas].sum()
    return pd.Series({
        "pct_hogares_imputados": 100 * g["alguna_imputacion"].astype(float).mean(),
        **{f"pct_pos{p}": 100 * (g[f"idimph_pos{p}"] == 1).mean() for p in ii.POSICIONES},
        "pct_itf_expandido_imputado_min": 100 * (w * g["monto_imputado_min"].astype(float))[ambas_validas].sum() / itf_validas,
        "pct_itf_expandido_imputado_max": 100 * (w * g["monto_imputado_max"].astype(float))[ambas_validas].sum() / itf_validas,
    })


magnitud_hist = hogares_hist.groupby(["ANO4", "TRIMESTRE"]).apply(_magnitud_trimestre, include_groups=False).reset_index()
magnitud_hist["era"] = "historica"
magnitud_reg = imputa_regular[["ANO4", "TRIMESTRE", "pct_hogares_deccfr12", "pct_hogares_pondih0"]].assign(era="regional_2016")
magnitud = pd.concat([magnitud_hist, magnitud_reg], ignore_index=True)
magnitud.to_csv(META / "imputacion_magnitud_trimestre.csv", index=False)
print("resumen por era (media y rango entre trimestres):")
display(pd.DataFrame({
    "historica: % hogares imputados": magnitud_hist["pct_hogares_imputados"].describe(),
    "regular: % DECCFR==12": magnitud_reg["pct_hogares_deccfr12"].describe(),
    "regular: % PONDIH==0": magnitud_reg["pct_hogares_pondih0"].describe(),
    "historica: % ITF imputado (min)": magnitud_hist["pct_itf_expandido_imputado_min"].describe(),
    "historica: % ITF imputado (max)": magnitud_hist["pct_itf_expandido_imputado_max"].describe(),
}).round(2))
display(magnitud_hist.round(2))

### Salto en 2009T3

Si las bases de 2003T3-2009T2 no estuvieran reprocesadas, la frecuencia de imputación
cambiaría de nivel en 2009T3.

In [ ]:
clave = magnitud_hist["ANO4"] * 10 + magnitud_hist["TRIMESTRE"]
antes = magnitud_hist[clave.between(20081, 20092)]
despues = magnitud_hist[clave.between(20093, 20104)]
print(f"% hogares imputados 2008T1-2009T2: {antes['pct_hogares_imputados'].mean():.2f} "
      f"(rango {antes['pct_hogares_imputados'].min():.2f}-{antes['pct_hogares_imputados'].max():.2f})")
print(f"% hogares imputados 2009T3-2010T4: {despues['pct_hogares_imputados'].mean():.2f} "
      f"(rango {despues['pct_hogares_imputados'].min():.2f}-{despues['pct_hogares_imputados'].max():.2f})")

### Distribución de `fraccion_min` / `fraccion_max` por tramo

**Población común**: hogares con las dos fracciones válidas (la misma para min y max, así
la comparación no mezcla poblaciones). Aparte, por trimestre, los hogares con
`fraccion_min` válida y `fraccion_max` NA, por causa, y los que tienen `ITF==0`.

In [ ]:
ambas_validas = hogares_hist["fraccion_min"].notna() & hogares_hist["fraccion_max"].notna()
comun = hogares_hist[ambas_validas].copy()
comun["tramo_min"] = ii.tramo_fraccion(comun["fraccion_min"])
comun["tramo_max"] = ii.tramo_fraccion(comun["fraccion_max"])

tramos = []
for lado in ("min", "max"):
    t = (comun.groupby(["ANO4", "TRIMESTRE"])[f"tramo_{lado}"].value_counts(normalize=True).mul(100)
         .unstack(fill_value=0).reindex(columns=list(ii.TRAMOS_FRACCION), fill_value=0))
    t["fraccion"] = lado
    tramos.append(t.reset_index())
tramos = pd.concat(tramos, ignore_index=True)
tramos.to_csv(META / "imputacion_tramos_trimestre.csv", index=False)
print("promedio entre trimestres (% de hogares, población común):")
display(tramos.groupby("fraccion")[list(ii.TRAMOS_FRACCION)].mean().round(2))

difiere = comun.assign(difiere=comun["tramo_min"] != comun["tramo_max"]).groupby(["ANO4", "TRIMESTRE"]).agg(
    n_hogares_comun=("difiere", "size"), pct_tramo_difiere=("difiere", "mean"))
difiere["pct_tramo_difiere"] *= 100
difiere["pct_tramo_difiere_entre_imputados"] = 100 * comun[comun["alguna_imputacion"] == True].assign(  # noqa: E712
    d=lambda x: x["tramo_min"] != x["tramo_max"]).groupby(["ANO4", "TRIMESTRE"])["d"].mean()

min_sin_max = hogares_hist[hogares_hist["fraccion_min"].notna() & hogares_hist["fraccion_max"].isna()]
causas = min_sin_max.assign(
    causa=np.select([min_sin_max["max_na_plan_sin_pj"], min_sin_max["max_na_menos9"]],
                    ["pos5_sin_pj", "menos9_en_monto_max"], default="otra")
).groupby(["ANO4", "TRIMESTRE"])["causa"].value_counts().unstack(fill_value=0)
na_por_trimestre = hogares_hist.groupby(["ANO4", "TRIMESTRE"]).agg(
    n_itf_cero=("itf_cero", "sum"), n_ambas_na=("fraccion_min", lambda s: int(s.isna().sum())))
calidad_fraccion = difiere.join(causas, how="left").join(na_por_trimestre).fillna(0)
calidad_fraccion.to_csv(META / "imputacion_calidad_fraccion_trimestre.csv")
display(calidad_fraccion.round(2))

## 4. Selectividad

Perfil de los hogares imputados contra los no imputados (era histórica) y de
`DECCFR==12` contra el resto (era regular). Los perfiles son conteos de hogares sin
ponderar en ambas eras (los ponderadores de las dos eras no son comparables, ver
`decisiones.md`). El quintil de `ITF` (ya imputado) se arma ponderado por `peso_hogar`
dentro de cada trimestre y solo en la histórica: en la regular el `ITF` de `DECCFR==12` es 0.

In [ ]:
def _quintil_ponderado(g: pd.DataFrame) -> pd.Series:
    orden = g["ITF"].astype(float).rank(method="first")
    w = g["peso_hogar"].astype(float).fillna(0).to_numpy()
    idx = np.argsort(orden.to_numpy())
    acumulado = np.empty(len(g))
    acumulado[idx] = np.cumsum(w[idx]) / w.sum()
    return pd.Series(np.minimum((acumulado * 5).astype(int) + 1, 5), index=g.index)


hogares_hist["quintil_itf"] = hogares_hist.groupby(["ANO4", "TRIMESTRE"], group_keys=False).apply(
    _quintil_ponderado, include_groups=False)
hogares_hist["grupo"] = np.where((hogares_hist["alguna_imputacion"] == True).fillna(False), "imputado", "no_imputado")  # noqa: E712
hogares_reg["grupo"] = np.where((hogares_reg["DECCFR"] == 12).fillna(False), "deccfr12", "resto")

quintil = pd.crosstab(hogares_hist["quintil_itf"], hogares_hist["grupo"], normalize="columns").mul(100)
situacion_hist = pd.crosstab(hogares_hist["situacion_laboral"], hogares_hist["grupo"], normalize="columns").mul(100)
situacion_reg = pd.crosstab(hogares_reg["situacion_laboral"], hogares_reg["grupo"], normalize="columns").mul(100)
independientes = pd.DataFrame({
    "historica_imputado": [100 * hogares_hist.loc[hogares_hist["grupo"] == "imputado", "hay_independiente"].mean()],
    "historica_no_imputado": [100 * hogares_hist.loc[hogares_hist["grupo"] == "no_imputado", "hay_independiente"].mean()],
    "regular_deccfr12": [100 * hogares_reg.loc[hogares_reg["grupo"] == "deccfr12", "hay_independiente"].mean()],
    "regular_resto": [100 * hogares_reg.loc[hogares_reg["grupo"] == "resto", "hay_independiente"].mean()],
}, index=["% hogares con algún independiente"])
tasa_aglo = pd.DataFrame({
    "historica_pct_imputados": hogares_hist.groupby("AGLOMERADO")["grupo"].apply(lambda s: 100 * (s == "imputado").mean()),
    "regular_pct_deccfr12": hogares_reg.groupby("AGLOMERADO")["grupo"].apply(lambda s: 100 * (s == "deccfr12").mean()),
})
print("quintil de ITF (histórica, % de cada grupo):"); display(quintil.round(1))
print("situación laboral (histórica):"); display(situacion_hist.round(1))
print("situación laboral (regular):"); display(situacion_reg.round(1))
display(independientes.round(1))
print(f"correlación entre aglomerados de % imputados (hist) y % DECCFR==12 (reg): "
      f"{tasa_aglo.corr().iloc[0, 1]:.2f}")
display(tasa_aglo.describe().round(1))
selectividad = pd.concat({"quintil_hist": quintil, "situacion_hist": situacion_hist, "situacion_reg": situacion_reg,
                          "independientes": independientes.T, "aglomerado": tasa_aglo}, names=["tabla", "categoria"])
selectividad.to_csv(META / "imputacion_selectividad.csv")

## 5. Efecto en el panel (en memoria, sin escribir capas)

Pares `h=1`/`h=4` de `03_panel` unidos a la distancia de `02b`, con
`trayectoria_distancia.construir_pares_distancia` (la distancia histórica no tiene
`PONDIH`: se le agrega `PONDIH=NA` en memoria, sin tocar `src/`). Grupo de referencia:
pares regulares (con `DECCFR==12` ya en NA por `ingreso_no_declarado`).

**Fracción por punta del par**: un hogar sin imputación cuenta con fracción 0 aunque su
`ITF` sea 0 (no tiene monto imputado); un hogar imputado toma su fracción solo si min y max
son válidas (población común), si no la punta queda NA. El tramo del par usa
`max(fraccion_t, fraccion_th)`.

In [ ]:
def _cargar_distancia(anio: int, trimestre: int) -> pd.DataFrame:
    df = pd.read_parquet(DISTANCIA / f"ANO4={anio}" / f"TRIMESTRE={trimestre}" / "data.parquet")
    if "PONDIH" not in df:
        df["PONDIH"] = pd.array([pd.NA] * len(df), dtype="Int32")
    return df


def construir_pares(trimestres: list[tuple[int, int]], h: int) -> pd.DataFrame:
    distancias, partes = {}, []
    for anio, trimestre in trimestres:
        if not panel.tiene_destino_valido(anio, trimestre, h):
            continue
        destino = panel.sumar_trimestres(anio, trimestre, h)
        p_panel = PANEL_DIR / f"pares_h{h}" / f"ANO4={anio}" / f"TRIMESTRE={trimestre}" / "data.parquet"
        if not p_panel.exists():
            continue
        for q in ((anio, trimestre), destino):
            if q not in distancias:
                distancias[q] = _cargar_distancia(*q)
        partes.append(trayectoria_distancia.construir_pares_distancia(
            distancias[(anio, trimestre)], distancias[destino], pd.read_parquet(p_panel)))
    pares = pd.concat(partes, ignore_index=True)
    pares["h"] = h
    return pares


fraccion_punta = hogares_hist[["CODUSU", "NRO_HOGAR", "ANO4", "TRIMESTRE", "alguna_imputacion"]].copy()
validas = hogares_hist["fraccion_min"].notna() & hogares_hist["fraccion_max"].notna()
no_imputado = (hogares_hist["alguna_imputacion"] == False).fillna(False).to_numpy()  # noqa: E712
for lado in ("min", "max"):
    valor = hogares_hist[f"fraccion_{lado}"].astype(float).where(validas)
    fraccion_punta[f"fraccion_{lado}"] = np.where(no_imputado, 0.0, valor)


def agregar_imputacion(pares: pd.DataFrame) -> pd.DataFrame:
    for sufijo in ("t", "th"):
        lado = fraccion_punta.rename(columns={c: f"{c}_{sufijo}" for c in fraccion_punta.columns if c != "CODUSU"})
        pares = pares.merge(lado, on=["CODUSU", f"NRO_HOGAR_{sufijo}", f"ANO4_{sufijo}", f"TRIMESTRE_{sufijo}"], how="left")
    pares["estado_imputacion"] = ii.clasificar_estado_imputacion_par(pares["alguna_imputacion_t"], pares["alguna_imputacion_th"])
    for lado in ("min", "max"):
        f = pares[[f"fraccion_{lado}_t", f"fraccion_{lado}_th"]].max(axis=1, skipna=False)
        pares[f"fraccion_{lado}_par"] = f
        pares[f"tramo_{lado}"] = ii.tramo_fraccion(f)
    return pares


t0 = time.time()
pares_hist = {h: agregar_imputacion(construir_pares(TRIMESTRES_HIST, h)) for h in (1, 4)}
pares_reg = {h: construir_pares(TRIMESTRES_REG, h) for h in (1, 4)}
for h in (1, 4):
    assert (pares_hist[h]["era_t"] == "historica").all() and (pares_hist[h]["era_th"] == "historica").all()
    assert (pares_reg[h]["era_t"] == "regional_2016").all() and (pares_reg[h]["era_th"] == "regional_2016").all()
    print(f"h={h}: {len(pares_hist[h]):,} pares históricos, {len(pares_reg[h]):,} regulares")
print(f"{time.time() - t0:.0f}s")

### Métricas por grupo

Por grupo: N con `par_valido_simetrica`, dispersión de `variacion_simetrica` (desvío, IQR,
p10/p90), y pendiente de regresión a la media (`variacion_log` sobre
`log_ratio_linea_pobreza_t`, con `par_valido_log`). Medias por decil de
`ratio_linea_pobreza_t` con **cortes comunes** a todos los grupos (pool de pares válidos
históricos + regulares de ese `h`).

In [ ]:
def cortes_decil_comunes(h: int) -> np.ndarray:
    pool = pd.concat([pares_hist[h].loc[pares_hist[h]["par_valido_simetrica"], "ratio_linea_pobreza_t"],
                      pares_reg[h].loc[pares_reg[h]["par_valido_simetrica"], "ratio_linea_pobreza_t"]]).astype(float)
    cortes = np.quantile(pool, np.linspace(0, 1, 11))
    cortes[0], cortes[-1] = -np.inf, np.inf
    return cortes


CORTES = {h: cortes_decil_comunes(h) for h in (1, 4)}


def metricas(pares: pd.DataFrame, h: int) -> dict:
    sim = pares[pares["par_valido_simetrica"]]
    log = pares[pares["par_valido_log"]]
    return {**ii.resumen_dispersion(sim["variacion_simetrica"]),
            "pendiente_regresion_media": ii.pendiente_regresion_media(log["log_ratio_linea_pobreza_t"], log["variacion_log"]),
            "n_par_valido_log": len(log)}


def medias_por_decil(pares: pd.DataFrame, h: int) -> pd.Series:
    sim = pares[pares["par_valido_simetrica"]]
    decil = pd.cut(sim["ratio_linea_pobreza_t"].astype(float), CORTES[h], labels=range(1, 11))
    return sim.groupby(decil, observed=False)["variacion_simetrica"].mean()


filas_grupo, filas_decil = [], []
for h in (1, 4):
    ref = pares_reg[h]
    filas_grupo.append({"h": h, "agrupacion": "referencia", "grupo": "regular_valido", **metricas(ref, h)})
    filas_decil.append(medias_por_decil(ref, h).rename((h, "referencia", "regular_valido")))
    filas_grupo.append({"h": h, "agrupacion": "historica_todos", "grupo": "todos", **metricas(pares_hist[h], h)})
    for estado, g in pares_hist[h].groupby("estado_imputacion"):
        filas_grupo.append({"h": h, "agrupacion": "estado", "grupo": estado, **metricas(g, h)})
        filas_decil.append(medias_por_decil(g, h).rename((h, "estado", estado)))
    comun_par = pares_hist[h][pares_hist[h]["fraccion_min_par"].notna() & pares_hist[h]["fraccion_max_par"].notna()]
    for lado in ("min", "max"):
        for tramo in ii.TRAMOS_FRACCION:
            g = comun_par[comun_par[f"tramo_{lado}"] == tramo]
            if len(g):
                filas_grupo.append({"h": h, "agrupacion": f"tramo_{lado}", "grupo": tramo, **metricas(g, h)})
                filas_decil.append(medias_por_decil(g, h).rename((h, f"tramo_{lado}", tramo)))
metricas_grupo = pd.DataFrame(filas_grupo)
deciles_grupo = pd.DataFrame(filas_decil)
deciles_grupo.index = pd.MultiIndex.from_tuples(deciles_grupo.index, names=["h", "agrupacion", "grupo"])
metricas_grupo.to_csv(META / "imputacion_panel_metricas_grupo.csv", index=False)
deciles_grupo.to_csv(META / "imputacion_panel_medias_decil.csv")
display(metricas_grupo.round(4))

In [ ]:
display(deciles_grupo.round(3))

### N y proporción de pares por estado y tramo, por `h` y trimestre de origen

In [ ]:
conteos_origen = []
for h in (1, 4):
    validos = pares_hist[h][pares_hist[h]["par_valido_simetrica"]]
    for col in ("estado_imputacion", "tramo_min", "tramo_max"):
        c = validos.groupby(["ANO4_t", "TRIMESTRE_t"])[col].value_counts(dropna=False).rename("n").reset_index()
        c["proporcion"] = c["n"] / c.groupby(["ANO4_t", "TRIMESTRE_t"])["n"].transform("sum")
        c = c.rename(columns={col: "grupo"})
        c["agrupacion"], c["h"] = col, h
        conteos_origen.append(c)
conteos_origen = pd.concat(conteos_origen, ignore_index=True)
conteos_origen.to_csv(META / "imputacion_panel_conteos_origen.csv", index=False)
print("proporción media entre orígenes (pares válidos):")
display(conteos_origen.groupby(["h", "agrupacion", "grupo"], dropna=False)["proporcion"].mean().unstack(0).round(3))

### Pares válidos bajo cada variante

- **A**: todos los pares históricos válidos.
- **C total**: ninguna punta imputada.
- **C-parcial u**: fracción del par (máx. de las dos puntas) ≤ u, con min y con max; las
  puntas imputadas sin fracción válida quedan afuera en las dos versiones.

Se compara contra los pares regulares válidos (lo que se suma a 2016-2025).

In [ ]:
filas_variante = []
for h in (1, 4):
    validos = pares_hist[h][pares_hist[h]["par_valido_simetrica"]]
    n_reg = int(pares_reg[h]["par_valido_simetrica"].sum())
    variantes = {"A": pd.Series(True, index=validos.index), "C_total": validos["estado_imputacion"] == "ninguna"}
    for u in UMBRALES:
        for lado in ("min", "max"):
            variantes[f"C_parcial_{u}_{lado}"] = (validos[f"fraccion_{lado}_par"] <= u).fillna(False)
    for nombre, mascara in variantes.items():
        n = int(mascara.sum())
        filas_variante.append({"h": h, "variante": nombre, "n_pares_historicos_validos": n,
                               "pct_de_A": 100 * n / len(validos), "n_pares_regulares_validos": n_reg,
                               "pct_que_suma_a_regulares": 100 * n / n_reg})
pares_por_variante = pd.DataFrame(filas_variante)
pares_por_variante.to_csv(META / "imputacion_pares_por_variante.csv", index=False)
display(pares_por_variante.round(1))

### Alrededor de 2009T3: dispersión de los pares sin imputar por origen (2008T1-2011T4)

In [ ]:
filas_ventana = []
for h in (1, 4):
    sin_imputar = pares_hist[h][(pares_hist[h]["estado_imputacion"] == "ninguna") & pares_hist[h]["par_valido_simetrica"]]
    for (anio, trimestre), g in sin_imputar.groupby(["ANO4_t", "TRIMESTRE_t"]):
        if (2008, 1) <= (anio, trimestre) <= (2011, 4):
            filas_ventana.append({"h": h, "origen": f"{anio}T{trimestre}", **ii.resumen_dispersion(g["variacion_simetrica"])})
ventana_2009 = pd.DataFrame(filas_ventana)
ventana_2009.to_csv(META / "imputacion_ventana_2009.csv", index=False)
display(ventana_2009.pivot(index="origen", columns="h", values=["desvio", "iqr"]).round(4))

## 6. Sensibilidad contra CEDLAS

Misma muestra comparable que 05c (sin aglomerados 38/91/93, peso de personas
`peso_hogar·n_miembros`, `cr.semestres_historicos_computables`). Variantes: **A** (imputados
adentro); **C1/C2** (todo hogar imputado como no declarado, sin / con reponderación
dentro de aglomerado × trimestre); **C-parcial** `u ∈ {0.1, 0.25, 0.5}` × {min, max} × {sin,
con reponderación} (excluye hogares con fracción > u o con fracción no válida, la misma
población para min y max). Se compara contra la serie comparable y, como segunda
referencia, contra la serie de CEDLAS sin corrección por no respuesta
(`*_sin_correccion_nr`).

In [ ]:
base_cedlas = hogares_hist[~hogares_hist["AGLOMERADO"].isin(AGLOMERADOS_EXCLUIDOS_COMPARABLE)].copy()
base_cedlas["peso_personas"] = base_cedlas["peso_hogar"].astype(float) * base_cedlas["n_miembros"].astype(float)
validas_cedlas = base_cedlas["fraccion_min"].notna() & base_cedlas["fraccion_max"].notna()
imputado_cedlas = (base_cedlas["alguna_imputacion"] == True).fillna(False)  # noqa: E712

exclusiones = {"A": pd.Series(False, index=base_cedlas.index), "C": imputado_cedlas}
for u in UMBRALES:
    for lado in ("min", "max"):
        exclusiones[f"C_parcial_{u}_{lado}"] = imputado_cedlas & (~validas_cedlas | (base_cedlas[f"fraccion_{lado}"] > u).fillna(True))

pesos_variante = {}
for nombre, excluir in exclusiones.items():
    if nombre == "A":
        pesos_variante["A"] = base_cedlas["peso_personas"]
        continue
    pesos_variante[f"{nombre}_sin_reponderar"] = base_cedlas["peso_personas"].where(~excluir, 0.0)
    pesos_variante[f"{nombre}_reponderado"] = ii.reponderar_dentro_de_celda(
        base_cedlas, "peso_personas", excluir, ["ANO4", "TRIMESTRE", "AGLOMERADO"])

cedlas = pd.read_csv(CEDLAS)
filas_cedlas = []
for anio, semestre, componentes, movil in cr.semestres_historicos_computables(TRIMESTRES_HIST):
    en_semestre = (base_cedlas["ANO4"] * 10 + base_cedlas["TRIMESTRE"]).isin([a * 10 + t for a, t in componentes])
    referencia = cedlas[(cedlas["ANO4"] == anio) & (cedlas["SEMESTRE"] == semestre)]
    if referencia.empty:
        continue
    referencia = referencia.iloc[0]
    for nombre, peso in pesos_variante.items():
        df = base_cedlas[en_semestre].assign(_w=peso[en_semestre])
        df = df[df["_w"] > 0]
        pobreza = distancia_linea_pobreza.tasa_ponderada(df, "pobre", "_w")
        indigencia = distancia_linea_pobreza.tasa_ponderada(df, "indigente", "_w")
        filas_cedlas.append({
            "ANO4": anio, "SEMESTRE": semestre, "variante": nombre, "pobreza": pobreza, "indigencia": indigencia,
            "brecha_pobreza_pp": pobreza - referencia["pob_serie_comparable"],
            "brecha_indigencia_pp": indigencia - referencia["indig_serie_comparable"],
            "brecha_pobreza_sin_corr_nr_pp": pobreza - referencia["pob_sin_correccion_nr"],
            "brecha_indigencia_sin_corr_nr_pp": indigencia - referencia["indig_sin_correccion_nr"]})
brechas_cedlas = pd.DataFrame(filas_cedlas)
brechas_cedlas.to_csv(META / "imputacion_variantes_vs_cedlas.csv", index=False)

columnas_brecha = ["brecha_pobreza_pp", "brecha_indigencia_pp", "brecha_pobreza_sin_corr_nr_pp",
                   "brecha_indigencia_sin_corr_nr_pp"]
resumen_cedlas = brechas_cedlas.groupby("variante")[columnas_brecha].agg(["mean", lambda s: s.abs().mean()])
resumen_cedlas.columns = [f"{c}_{'media' if a == 'mean' else 'media_abs'}" for c, a in resumen_cedlas.columns]
resumen_cedlas = resumen_cedlas.reindex(list(pesos_variante))
resumen_cedlas.to_csv(META / "imputacion_variantes_vs_cedlas_resumen.csv")
print(f"{brechas_cedlas[['ANO4', 'SEMESTRE']].drop_duplicates().shape[0]} semestres comparados")
display(resumen_cedlas.round(2))

## 7. Cierre — tres tablas lado a lado

In [ ]:
tabla_i = pares_por_variante.pivot(index="variante", columns="h", values="n_pares_historicos_validos")
tabla_i.columns = [f"historica_h{h}" for h in tabla_i.columns]
for h in (1, 4):
    tabla_i[f"regular_h{h}"] = int(pares_reg[h]["par_valido_simetrica"].sum())
tabla_i = tabla_i.reindex(pares_por_variante["variante"].drop_duplicates())
print("(i) N de pares válidos (par_valido_simetrica) por variante y umbral, por era")
display(tabla_i)

tabla_ii = metricas_grupo[metricas_grupo["agrupacion"].isin(["referencia", "historica_todos", "estado", "tramo_min", "tramo_max"])]
tabla_ii = tabla_ii.pivot_table(index=["agrupacion", "grupo"], columns="h",
                                values=["n", "desvio", "iqr", "pendiente_regresion_media"], sort=False)
print("(ii) dispersión y pendiente por estado y por tramo de fracción imputada (población común min/max), "
      "más la referencia regular")
display(tabla_ii.round(4))

tabla_iii = resumen_cedlas[["brecha_pobreza_pp_media", "brecha_pobreza_pp_media_abs",
                            "brecha_indigencia_pp_media", "brecha_indigencia_pp_media_abs"]]
print("(iii) brechas contra CEDLAS (serie comparable) por variante y umbral, en pp")
display(tabla_iii.round(2))

### ¿La conclusión cambia entre `fraccion_min` y `fraccion_max`?

Para cada umbral: pares que quedan, pendiente y dispersión de los pares que quedan
(`h=4`), y brecha absoluta media contra CEDLAS (reponderado), con min y con max.

In [ ]:
filas_robustez = []
for u in UMBRALES:
    fila = {"umbral": u}
    for lado in ("min", "max"):
        quedan = pares_hist[4][(pares_hist[4][f"fraccion_{lado}_par"] <= u).fillna(False)]
        m = metricas(quedan, 4)
        fila[f"n_h4_{lado}"] = m["n"]
        fila[f"desvio_h4_{lado}"] = m["desvio"]
        fila[f"pendiente_h4_{lado}"] = m["pendiente_regresion_media"]
        fila[f"brecha_abs_pobreza_{lado}"] = resumen_cedlas.loc[f"C_parcial_{u}_{lado}_reponderado", "brecha_pobreza_pp_media_abs"]
        fila[f"brecha_abs_indigencia_{lado}"] = resumen_cedlas.loc[f"C_parcial_{u}_{lado}_reponderado", "brecha_indigencia_pp_media_abs"]
    filas_robustez.append(fila)
robustez = pd.DataFrame(filas_robustez)
robustez["mismo_lado_umbral_1pp_pobreza"] = (robustez["brecha_abs_pobreza_min"] <= 1) == (robustez["brecha_abs_pobreza_max"] <= 1)
robustez["mismo_lado_umbral_1pp_indigencia"] = (robustez["brecha_abs_indigencia_min"] <= 1) == (robustez["brecha_abs_indigencia_max"] <= 1)
robustez.to_csv(META / "imputacion_robustez_min_max.csv", index=False)
display(robustez.round(4))

## Anomalías encontradas

1. **`PP08J*` (aguinaldo, bonificaciones no habituales, retroactivos) van por fuera de `P21`**
   [confirmado, 47 trimestres] — **resuelta** en `src/imputacion_ingresos.py`. En la primera
   corrida de este notebook I1a incluía `PP08J*` dentro de `P21` y fallaba en ~20% de las
   personas con pos. 1 dígitos 1-3 en T1/T3 (meses con aguinaldo; 1-2% en T2/T4). Evidencia:
   con `PP08J1>0`, 0% lo tiene dentro de `P21` y 86-100% (mediana ~97%) lo tiene en `P47T` por
   fuera de `P21`; con `PP08J2+PP08J3>0` y `PP08J1==0`, 100% cumple `P21 = D1+F1+F2` y 98,6-100%
   los tiene en `P47T` por fuera. Corrección: I1a = `P21 == PP08D1+PP08F1+PP08F2` e
   `INCLUSIONES["P21"]` sin `PP08J*`. Con eso I1a se cumple en el **100%** de esas personas en
   todos los trimestres y el respaldo `P21` no se usa nunca. I2 también suma `PP08J*`
   (`P47T == P21+PP08J1+PP08J2+PP08J3+TOT_P12+T_VI`): con la versión sin `PP08J*` fallaba en
   ~6% de las personas en T1/T3; con la corrección se cumple en 99,5-100% de las personas en
   todos los trimestres, sin diferencia entre T1/T3 y T2/T4.
2. **Pos. 4, dígitos 4-7 nunca aparecen** [confirmado]: el residuo "otros no laborales" no se usa.
3. Posición marcada con monto mapeado 0: 235 personas (pos. 1), 71 (pos. 4) y 5 (pos. 5) en todo el
   período. No hay `-9` ni `NA`. `IDIMPH` coincide con el agregado de `IDIMPP` (0 discrepancias).
4. **Pos. 5 sin `PJ*`**: 318 personas (`fraccion_max` NA). Marginal: el máximo es 0,133% de los
   hogares (2011T2).
5. `-7`/`-8` solo en `PP06C`/`PP06D`, como define la p. 30.
6. **`IMPUTA` (regular)**: ~50-160 personas por trimestre. `NaN` reemplaza a `0` desde 2020T4.
   El 60% de los hogares con `IMPUTA` son `DECCFR==12`. No es la marca de no respuesta de la era
   regular, y qué imputa no está documentado [no determinado].
7. **2009T3 sin quiebre**: la imputación sube de forma gradual de ~14% (2006) a ~24-25%
   (2011-2012). Las capturas de Wayback del caché (2014-09-08 y 2017-10-18) son posteriores al
   reprocesamiento [inferido: versiones reprocesadas].

## Evidencia (después de la corrección de `PP08J*`)

- **Magnitud** [confirmado]: 20,5% de hogares imputados en promedio (13,9-26,2%), contra 20,5% de
  `DECCFR==12` en la regular. El ITF expandido imputado está entre 20,1% (min) y 21,3% (max). En la
  población común, 13-14% de los hogares tienen más de la mitad del `ITF` imputado, y el tramo
  difiere entre min y max en 2% de los hogares (10% de los imputados).
- **Selectividad** [confirmado]: los imputados son más ricos y más formales o independientes, el
  mismo perfil que `DECCFR==12` (correlación entre aglomerados 0,76).
- **Panel** [confirmado]: pendiente de regresión a la media en h=1 de −0,19 sin imputar, contra
  −0,34/−0,40/−0,42 con imputación (solo t / solo t+h / ambas) y −0,22 en los regulares válidos.
  Por tramo de `fraccion_max`, la pendiente y el IQR crecen de forma monótona; el desvío no en los
  tramos bajos. El salto está en (0,5,1].
- **CEDLAS** [confirmado]: A ajusta mejor a la serie comparable (0,80/0,49pp de brecha absoluta en
  pobreza/indigencia); C ajusta mejor a la serie sin corrección por no respuesta.
- **min contra max**: misma conclusión en los tres umbrales, salvo indigencia con u=0,1, que queda
  al borde de 1pp (1,03 contra 0,97).

## Decisión tomada

**Trayectorias (variable objetivo): variante C total.** Todo hogar con `IDIMPH != 00000` se trata
como ingreso no declarado a efectos de la trayectoria. Se implementa **marcando**
(`par_valido_c` en `06c`), sin anular los ratios. **Tasas y niveles: variante A.** Ver
`.claude/decisiones.md`.